In [ ]:
# LLM batch prepation: items (unstructured data) ---> Create batches ---> run batches ---> fetch batches
# Create batches: Create parent folders of batches to be processed and store file names
# Run batches: create JSONL file ---> upload JSONL file ---> create batch 
# Fetch batches: retrieve batch ---> Download items ---> retrieve LLM response ---> items (structured data)
# Imports
import os
from dotenv import load_dotenv
import sys
import json
import pickle
import time
from pathlib import Path
from openai import OpenAI
from huggingface_hub import login
from datasets import load_dataset
from tqdm import tqdm
# Add the utils folder to Python path
notebook_dir = os.path.dirname(os.path.abspath("__file__"))
pricer_path = os.path.join(notebook_dir, "../utils")
sys.path.insert(0, pricer_path)
# Class Item to validate the returned data and has lots of helper functions
from pricer.items import Item
from litellm import completion

In [2]:
# Load environment variables in a file called .env
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
# Check the keys
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")  

OpenAI API Key exists and begins sk-proj-


In [ ]:
# Connect to OpenAI client library
openai = OpenAI()

In [4]:
# Login to Hugging Face
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [5]:
# Download the cleaned dataset from Hugging Face 
username = "omarshaarawy11"
dataset = f"{username}/best_deal_02_cleaned_data"
train, val, test = Item.from_hub(dataset)
items = train + val + test
print(f"Loaded {len(items):,} items")

Loaded 21,091 items


In [6]:
# Invistigate
items[0]

<Lurrose 100Pcs Full Cover Fake Toenails Artificial Transparent Nail Tips Nail Art for DIY = $6.99>

In [7]:
# Indexing items
for index, item in enumerate(items):
    item.id = index

In [8]:
# Select all columns 
columns = [
    'title', 
    'category', 
    'price',
    'full',
    'weight',
    'id'
]
for i in columns:
    attr = getattr(items[3], i)
    print(attr)

Garnier Fructis Color Sealer, Instant, Lightweight Leave-In, Color Shield, For Color-Treated Hair, 6 oz.
all beauty
24.0
Garnier Fructis Color Sealer, Instant, Lightweight Leave-In, Color Shield, For Color-Treated Hair, 6 oz.
['Proven to stop dry-out. Fight fade-out. The lightweight formula, infused with acai berry and grape seed oil, instantly seals in color for longer-lasting vibrancy while improving manageability and delivering lasting softness. Uva and uvb protectant help protect hair. Garnier fructis color shield instant color sealer lightweight leave-in instantly seals in color and conditions for extra softness.']
['Seals In Color And Conditions For Extra Softness', 'Infused With Acai Berry And Grape Seed Oil', 'For Color Treated Hair', 'Lightweight Formula']
{"Is Discontinued By Manufacturer": "No", "Product Dimensions": "1.35 x 2.39 x 6.63 inches; 7 Ounces", "UPC": "", "Manufacturer": "Garnier LLC"}
0.0
3


In [9]:
# Configurations
GENERATOR_MODEL = "gpt-4.1-nano"
# Input folder to LLM
BATCHES_FOLDER = "batches"
# Output folder from LLM
OUTPUT_FOLDER = "output"
BATCH_SIZE = 1_000

In [10]:
# Create folders to have batches later
# Parent folder
parent_folder = Path("best_deal_batches")
# Path of input folder
batches = parent_folder / BATCHES_FOLDER
# Path of output folder
output = parent_folder / OUTPUT_FOLDER
# Create parent directories if they do not exist
batches.mkdir(parents=True, exist_ok=True)
output.mkdir(parents=True, exist_ok=True)

In [11]:
# System prompt 
SYSTEM_PROMPT = """Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features"""

In [49]:
# Test cost per items to decide the number of batches
GENERATOR_MODEL = "gpt-4.1-nano"
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = completion(messages=messages, model=GENERATOR_MODEL)
print(response.choices[0].message.content)
print()
# Input tokens
print(f"Input tokens: {response.usage.prompt_tokens}")
# Output tokens
print(f"Output tokens: {response.usage.completion_tokens}")
# Cost converted to cents
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")

Title: 100Pcs Full Cover Fake Toenails for DIY Nail Art
Category: Beauty & Personal Care
Brand: Lurrose
Description: Durable, lightweight fake toenails designed for easy customization and natural fit.
Details: Made of eco-friendly ABS material, these nails are non-toxic, breathable, and easily trimmed to your preferred length and shape.

Input tokens: 441
Output tokens: 72
Cost: 0.007 cents


In [50]:
# Create batches
def create(items):
    batch_files = []
    for start in range(0, len(items), BATCH_SIZE):
        end = min(start + BATCH_SIZE, len(items))
        batch_files.append({
            "items": items,
            "start": start,
            "end": end,
            "filename": batches / f"{start}_{end}.jsonl",
        })
    print(f"Created {len(batch_files)} batches")
    return batch_files

In [51]:
batches_files = create(items)

Created 22 batches


In [52]:
# Run batches
def make_jsonl(item):
    body = {"model": GENERATOR_MODEL, 
            "messages": [{"role": "system", "content": SYSTEM_PROMPT}, 
                         {"role": "user", "content": item.full}
                        ]
            }
    line = {"custom_id": str(item.id), 
            "method": "POST", 
            "url": "/v1/chat/completions", 
            "body": body
            }
    return json.dumps(line)

In [53]:
# Test with single datapoint
make_jsonl(items[0])

'{"custom_id": "0", "method": "POST", "url": "/v1/chat/completions", "body": {"model": "gpt-4.1-nano", "messages": [{"role": "system", "content": "Create a concise description of a product. Respond only in this format. Do not include part numbers.\\nTitle: Rewritten short precise title\\nCategory: eg Electronics\\nBrand: Brand name\\nDescription: 1 sentence description\\nDetails: 1 sentence on features"}, {"role": "user", "content": "Lurrose 100Pcs Full Cover Fake Toenails Artificial Transparent Nail Tips Nail Art for DIY\\n[\'Description\', \'The false toenails are durable with perfect length. You have the option to wear them long or clip them short, easy to trim and file them to in any length and shape you like. Plus, ABS is kind of green enviromental material, and makes the nails durable, breathable, light even no pressure on your own toenails. Fit well to your natural toenails. Non toxic, no smell, no harm to your health.\', \'Feature\', \'- Color: As Shown.- Material: ABS.- Size: 

In [54]:
# Create JSONL file
def make_file(start, end, filename):
    batch_file = batches / filename
    with batch_file.open("w") as f:
            for item in items[start : end]:
                f.write(make_jsonl(item))
                f.write("\n")

In [55]:
# Test with single datapoint
make_file(0, 1, "test.jsonl")

In [56]:
# Batch mode 
# Upload JSONL file
def send_file(batches, filename):
    batch_file = batches / filename
    with batch_file.open("rb") as f:
        response = openai.files.create(
                    file=f,
                    purpose="batch"
                                    )
    file_id = response.id
    return file_id

In [57]:
# Test with single datapoint
file_id = send_file(batches, "test.jsonl")

In [58]:
# Create batch
def submit_batch(file_id):
    response = openai.batches.create(
                input_file_id=file_id,
                endpoint="/v1/chat/completions",
                completion_window="24h"
            )
    batch_id = response.id
    return batch_id

In [59]:
# Test with single datapoint
batch_id = submit_batch(file_id)

In [60]:
# Retrieve batch
def is_ready(batch_id, max_tokens = False):
    pbar = tqdm(desc="Processing batch", bar_format="{desc}: {elapsed}")
    # Backup old batch with 1000 items
    if max_tokens:
        # Use this in case of tokens limits accessed
        batch_id = "batch_6a3bd6488b1c8190a72a4f937c7e71b5"
    while True:
        batch = openai.batches.retrieve(batch_id)
        pbar.set_description(f"Status: {batch.status}")
        if batch.status == "completed":
            output_file_id = batch.output_file_id
            pbar.close()
            print('Done')
            return output_file_id
        if batch.status in ("failed", "cancelled", "expired"):
            pbar.close()
            print(batch.errors)
            raise RuntimeError(batch.status)
        time.sleep(5)

In [61]:
# Test with single datapoint
output_file_id = is_ready(batch_id)

Status: completed: : 01:06  

Done


In [62]:
# Download items
def fetch_output(output, filename, output_file_id):
    output_file = str(output / filename)
    response = openai.files.content(output_file_id)
    with open(output_file, "wb") as f:
        f.write(response.read())

In [63]:
# Test with single datapoint
results = fetch_output(output, "test.jsonl", output_file_id)

In [13]:
# Preprocessing items 
# Summary field: content of LLM response
def apply_output(output, filename):
    output_file = str(output / filename)
    with open(output_file, "r") as f:
        for line in f:
            json_line = json.loads(line)
            # Make the id as integer
            id = int(json_line["custom_id"])
            summary = json_line["response"]["body"]["choices"][0]["message"]["content"]
            items[id].summary = summary
    return True

In [14]:
# Test with single datapoint
summary = apply_output(output, "test.jsonl")

In [15]:
# Unsctructed data 
print(items[0].full)

Lurrose 100Pcs Full Cover Fake Toenails Artificial Transparent Nail Tips Nail Art for DIY
['Description', 'The false toenails are durable with perfect length. You have the option to wear them long or clip them short, easy to trim and file them to in any length and shape you like. Plus, ABS is kind of green enviromental material, and makes the nails durable, breathable, light even no pressure on your own toenails. Fit well to your natural toenails. Non toxic, no smell, no harm to your health.', 'Feature', '- Color: As Shown.- Material: ABS.- Size: 14.3 x 7.2 x 1cm.', 'Package Including', '100 x Pieces fake toenails']
['The false toenails are durable with perfect length. You have the option to wear them long or clip them short, easy to trim and file them to in any length and shape you like.', 'ABS is kind of green enviromental material, and makes the nails durable, breathable, light even no pressure on your own nails.', 'Fit well to your natural toenails. Non toxic, no smell, no harm to 

In [16]:
# Structured data
print(items[0].summary)

Title: Lurrose 100pcs Full Cover Fake Toenails  
Category: Nail Art & Accessories  
Brand: Lurrose  
Description: High-quality, durable artificial toenails perfect for DIY nail art and manicures.  
Details: Features 100 lightweight, eco-friendly ABS nails that are easy to trim, shape, and fit comfortably to natural toenails, suitable for various styles and perfect as a gift.


In [17]:
# Push dataset to Hugging Face
import truststore
truststore.inject_into_ssl()
username = "omarshaarawy11"
items_data = f"{username}/best_deal_03_batches_test"
# Upload only a copy of the first item so the full dataset stays unchanged
first_item = items[0].model_copy(deep=True)
first_item.full = None
first_item.id = None
Item.push_to_hub(items_data, [first_item], [first_item], [first_item])

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


Upload 0 LFS files: 0it [00:00, ?it/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


Upload 0 LFS files: 0it [00:00, ?it/s]

No files have been modified since last commit. Skipping to prevent empty commit.


In [33]:
# Inspect batches files store
batches_files

[{'items': [<Lurrose 100Pcs Full Cover Fake Toenails Artificial Transparent Nail Tips Nail Art for DIY = $6.99>,
   <BioMiracle StarDust Pixie Bubble Mask, Clarifying Foaming Face Mask with Green Tea and Apple, Carbonated Bubble Cupro Sheet Mask for Clear, Even Skin = $5.99>,
   <VIROCHEMISTRY Pheromones For Women (Elixir) - Elegant, Ultra Strength Organic Fragrance Body Perfume (1 Fl. Oz) = $29.8>,
   <Garnier Fructis Color Sealer, Instant, Lightweight Leave-In, Color Shield, For Color-Treated Hair, 6 oz. = $24.0>,
   <Enjoy VOLUMIZING ELIXIR, Style (with Sleek Steel Pin Tail Comb) (8.8 oz) = $22.49>,
   <Bulk Herbs: Clay, White Kaolin = $2.5>,
   <Armáf Täg Hïm Colognė For Men 3.4 oz Eau De Toilette Spray = $24.84>,
   <Wholesale CASE of 10 - GOJO Supro Max Cherry Hand Cleaner-Supro Max Hand Cleaner, 2000l, Cherry = $136.17>,
   <Amope PediMask Kit- 20 Minute Foot Mask to Rejuvenate and Soothe Your Feet with Blend of Moisturizers and Macadamia Oils for Baby Smooth Feet in Minutes (Pa

In [34]:
batches_files[0]

{'items': [<Lurrose 100Pcs Full Cover Fake Toenails Artificial Transparent Nail Tips Nail Art for DIY = $6.99>,
  <BioMiracle StarDust Pixie Bubble Mask, Clarifying Foaming Face Mask with Green Tea and Apple, Carbonated Bubble Cupro Sheet Mask for Clear, Even Skin = $5.99>,
  <VIROCHEMISTRY Pheromones For Women (Elixir) - Elegant, Ultra Strength Organic Fragrance Body Perfume (1 Fl. Oz) = $29.8>,
  <Garnier Fructis Color Sealer, Instant, Lightweight Leave-In, Color Shield, For Color-Treated Hair, 6 oz. = $24.0>,
  <Enjoy VOLUMIZING ELIXIR, Style (with Sleek Steel Pin Tail Comb) (8.8 oz) = $22.49>,
  <Bulk Herbs: Clay, White Kaolin = $2.5>,
  <Armáf Täg Hïm Colognė For Men 3.4 oz Eau De Toilette Spray = $24.84>,
  <Wholesale CASE of 10 - GOJO Supro Max Cherry Hand Cleaner-Supro Max Hand Cleaner, 2000l, Cherry = $136.17>,
  <Amope PediMask Kit- 20 Minute Foot Mask to Rejuvenate and Soothe Your Feet with Blend of Moisturizers and Macadamia Oils for Baby Smooth Feet in Minutes (Pack of 3) 

In [35]:
len(batches_files)

22

In [38]:
# Work with the whole dataset one batch at a time
# Run and fetch batches with retry support
def run(batches_files, max_retries=5, retry_wait=60):
    batch_ids = []
    for batch_number, batch in enumerate(tqdm(batches_files), start=1):
        filename = batch["filename"].name
        make_file(batch["start"], batch["end"], filename)

        for attempt in range(1, max_retries + 1):
            try:
                file_id = send_file(batches, filename)
                batch_id = submit_batch(file_id)
                output_file_id = is_ready(batch_id)
                fetch_output(output, filename, output_file_id)
                apply_output(output, filename)
                batch_ids.append(batch_id)
                print(f"Completed batch {batch_number}/{len(batches_files)}")
                break
            except RuntimeError as error:
                if attempt == max_retries:
                    raise RuntimeError(
                        f"Batch {batch_number} failed after {max_retries} attempts: {error}"
                    ) from error
                print(
                    f"Batch {batch_number} failed ({error}). "
                    f"Waiting {retry_wait} seconds before retry {attempt + 1}/{max_retries}."
                )
                time.sleep(retry_wait)

    print(f"Completed {len(batch_ids)} batches")
    return batch_ids


In [ ]:
batch_ids = run(batches_files)

In [25]:
# Push dataset to Hugging Face
import truststore
truststore.inject_into_ssl()
username = "omarshaarawy11"
items_data = f"{username}/best_deal_03_batches"
for item in items:
    item.full = None
    item.id = None
percentage = int(len(items) / 100)
train = items[:percentage * 80]
val = items[percentage * 80: percentage * 90]
test = items[percentage * 90: ]
Item.push_to_hub(items_data, train, val, test)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/17 [00:00<?, ?ba/s]

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


README.md:   0%|          | 0.00/747 [00:00<?, ?B/s]

c:\Users\EGShaaraOm\AppData\Local\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\EGShaaraOm\.cache\huggingface\hub\datasets--omarshaarawy11--best_deal_03_batches. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
